In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
PROJECT_ROOT = Path.cwd().parents[1]
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT

PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction')

In [2]:
ipo = pd.read_csv(DATA_RAW / "ipo_master.csv")

ipo.columns

Index(['COMPANY NAME', 'SECURITY TYPE', 'ISSUE PRICE', 'Symbol',
       'ISSUE START DATE', 'ISSUE END DATE', 'PRICE RANGE', 'DATE OF LISTING'],
      dtype='object')

In [3]:
ipo["listing_date"] = pd.to_datetime(
    ipo["DATE OF LISTING"], utc=True, errors="coerce"
)

ipo["company_name"] = (
    ipo["COMPANY NAME"]
    .str.lower()
    .str.strip()
)

ipo = ipo[["company_name", "listing_date", "Symbol"]].dropna()

ipo.shape

/var/folders/fk/0kqynd_95d71j1htw9b057j00000gn/T/ipykernel_20500/3264409420.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ipo["listing_date"] = pd.to_datetime(


(1179, 3)

In [4]:
sector_map = pd.read_csv(DATA_RAW / "company_sector_map.csv")

sector_map["symbol"] = sector_map["symbol"].str.upper().str.strip()
sector_map["sector"] = sector_map["sector"].str.lower().str.strip()

sector_map.head()

,symbol,sector


In [5]:
ipo_sector = ipo.merge(
    sector_map,
    left_on="Symbol",
    right_on="symbol",
    how="left"
)

ipo_sector["sector"].value_counts(dropna=False).head()

sector
NaN    1179
Name: count, dtype: int64

In [6]:
ipo_sector["sector"] = ipo_sector["sector"].fillna("unknown")

ipo_sector["sector"].value_counts().head()

sector
unknown    1179
Name: count, dtype: int64

In [7]:
market = pd.read_csv(
    DATA_PROCESSED / "market_features.csv",
    parse_dates=["listing_date"]
)

market.head()

,company_name,listing_date,market_return_1d,market_return_7d,market_return_30d,market_vol_30d,recent_market_trend,bull_market,bear_market,high_volatility,days_from_market_start
0,quick heal technologies limited,2016-02-18 00:00:00+00:00,0.007637,-0.048223,-0.127783,0.016474,0.079560,0,1,1,48
1,national highways authority of india,2016-03-11 00:00:00+00:00,-0.006852,0.012251,-0.026917,0.016950,0.039167,0,0,1,70
2,national highways authority of india,2016-03-11 00:00:00+00:00,-0.006852,0.012251,-0.026917,0.016950,0.039167,0,0,1,70
3,healthcare global enterprises limited,2016-03-30 00:00:00+00:00,0.028808,0.040511,0.045007,0.017181,-0.004496,0,0,1,89
4,bharat wire ropes limited,2016-04-01 00:00:00+00:00,0.000820,0.016414,0.034657,0.016469,-0.018243,0,0,0,91


In [8]:
ipo_full = ipo_sector.merge(
    market,
    on="listing_date",
    how="left"
)

ipo_full.shape

(3577, 15)

In [9]:
def market_regime(ret_30d):
    if ret_30d >= 0.05:
        return "bull"
    elif ret_30d <= -0.05:
        return "bear"
    else:
        return "neutral"

ipo_full["market_regime"] = ipo_full["market_return_30d"].apply(market_regime)

ipo_full["market_regime"].value_counts()

market_regime
neutral    1882
bull       1432
bear        263
Name: count, dtype: int64

In [10]:
vol_thresh = ipo_full["market_vol_30d"].quantile(0.75)

ipo_full["high_volatility"] = (
    ipo_full["market_vol_30d"] >= vol_thresh
).astype(int)

In [11]:
sector_density = (
    ipo_full
    .groupby("sector")
    .size()
    .rename("sector_ipo_count")
    .reset_index()
)

ipo_full = ipo_full.merge(
    sector_density,
    on="sector",
    how="left"
)

ipo_full["sector_ipo_count"].describe()

count    3577.0
mean     3577.0
std         0.0
min      3577.0
25%      3577.0
50%      3577.0
75%      3577.0
max      3577.0
Name: sector_ipo_count, dtype: float64

In [12]:
hot_sector_threshold = ipo_full["sector_ipo_count"].quantile(0.75)

ipo_full["is_hot_sector"] = (
    ipo_full["sector_ipo_count"] >= hot_sector_threshold
).astype(int)

ipo_full["is_hot_sector"].mean()

np.float64(1.0)

In [13]:
ipo_full["sector_x_bull"] = (
    (ipo_full["is_hot_sector"] == 1) &
    (ipo_full["market_regime"] == "bull")
).astype(int)

ipo_full["sector_x_bear"] = (
    (ipo_full["is_hot_sector"] == 1) &
    (ipo_full["market_regime"] == "bear")
).astype(int)

ipo_full["sector_x_high_vol"] = (
    (ipo_full["is_hot_sector"] == 1) &
    (ipo_full["high_volatility"] == 1)
).astype(int)

In [16]:
ipo_full.columns.tolist()

['company_name_x',
 'listing_date',
 'Symbol',
 'symbol',
 'sector',
 'company_name_y',
 'market_return_1d',
 'market_return_7d',
 'market_return_30d',
 'market_vol_30d',
 'recent_market_trend',
 'bull_market',
 'bear_market',
 'high_volatility',
 'days_from_market_start',
 'market_regime',
 'sector_ipo_count',
 'is_hot_sector',
 'sector_x_bull',
 'sector_x_bear',
 'sector_x_high_vol']

In [17]:
# Resolve merge suffixes cleanly
ipo_full = ipo_full.rename(
    columns={"company_name_x": "company_name"}
)

# Drop redundant name column
ipo_full = ipo_full.drop(columns=["company_name_y"])

# (Optional but recommended sanity check)
ipo_full[["company_name"]].head()

,company_name
0,indo farm equipment limited
1,anya polytech & fertilizers limited
2,unimech aerospace and manufacturing limited
3,carraro india limited
4,carraro india limited


In [18]:
sector_features = ipo_full[
    [
        "company_name",
        "listing_date",
        "sector",
        "is_hot_sector",
        "sector_ipo_count",
        "market_regime",
        "high_volatility",
        "sector_x_bull",
        "sector_x_bear",
        "sector_x_high_vol"
    ]
]

sector_features.head()

,company_name,listing_date,sector,is_hot_sector,sector_ipo_count,market_regime,high_volatility,sector_x_bull,sector_x_bear,sector_x_high_vol
0,indo farm equipment limited,2025-01-07 00:00:00+00:00,unknown,1,3577,bull,1,1,0,1
1,anya polytech & fertilizers limited,2025-01-02 00:00:00+00:00,unknown,1,3577,bull,1,1,0,1
2,unimech aerospace and manufacturing limited,2024-12-31 00:00:00+00:00,unknown,1,3577,bull,1,1,0,1
3,carraro india limited,2024-12-30 00:00:00+00:00,unknown,1,3577,bull,1,1,0,1
4,carraro india limited,2024-12-30 00:00:00+00:00,unknown,1,3577,bull,1,1,0,1


In [19]:
OUT_PATH = DATA_PROCESSED
OUT_PATH.mkdir(exist_ok=True, parents=True)

sector_features.to_csv(
    OUT_PATH / "sector_and_regime_features.csv",
    index=False
)

print("Saved:", OUT_PATH / "sector_and_regime_features.csv")

Saved: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/processed/sector_and_regime_features.csv
